In [ ]:
from pathlib import Path
print("Python runtime ready")
print("Working directory:", Path.cwd())

Python runtime ready
Working directory: /content


In [ ]:
%%writefile ecommerce_app.py
"""EduCommerce: in-memory application for practicing checkout and automated testing."""
from dataclasses import dataclass, field
from decimal import Decimal, ROUND_HALF_UP
from typing import Optional

D=lambda x: Decimal(str(x))

def money(x): return D(x).quantize(D('1'), rounding=ROUND_HALF_UP)

@dataclass
class Product:
    sku:str; title:str; price:Decimal; stock:int; active:bool=True
@dataclass
class Customer:
    customer_id:str; authenticated:bool=True; used_codes:set=field(default_factory=set)
@dataclass
class CartLine:
    sku:str; quantity:int
@dataclass
class Voucher:
    code:str; percent:int; min_subtotal:Decimal; cap:Decimal; active:bool=True
@dataclass
class Order:
    order_id:str; customer_id:str; total:Decimal; status:str; lines:list; discount:Decimal; shipping:Decimal; payment_method:str

class Catalog:
    def __init__(self):
        self.products={'A':Product('A','Wireless Mouse',D(200000),20),
                       'B':Product('B','Mechanical Keyboard',D(300000),15),
                       'C':Product('C','USB-C Hub',D(150000),12)}
    def get(self,sku):
        p=self.products.get(sku)
        if not p or not p.active: raise ValueError('PRODUCT_UNAVAILABLE')
        return p

class Cart:
    def __init__(self,catalog): self.catalog=catalog; self.lines=[]
    def add(self,sku,quantity):
        self.catalog.get(sku)
        if not isinstance(quantity,int) or quantity<1: raise ValueError('INVALID_QUANTITY')
        existing=next((x for x in self.lines if x.sku==sku),None)
        if existing: existing.quantity+=quantity
        else: self.lines.append(CartLine(sku,quantity))
    def subtotal(self): return sum((self.catalog.get(x.sku).price*x.quantity for x in self.lines),D(0))
    def total_quantity(self): return sum((line.quantity for line in self.lines),0)

class Inventory:
    def __init__(self,catalog): self.catalog=catalog
    def reserve(self,lines):
        for line in lines:
            if self.catalog.get(line.sku).stock < line.quantity: raise ValueError('OUT_OF_STOCK')
        for line in lines: self.catalog.get(line.sku).stock-=line.quantity
    def release(self,lines):
        for line in lines: self.catalog.get(line.sku).stock+=line.quantity

class Promotions:
    def __init__(self): self.codes={'SAVE10':Voucher('SAVE10',10,D(500000),D(100000))}
    def discount(self, code, subtotal, customer):
        if not code: return D(0)
        v=self.codes.get(code)
        if not v or not v.active: raise ValueError('INVALID_VOUCHER')
        if not customer.authenticated: raise ValueError('LOGIN_REQUIRED')
        if code in customer.used_codes: raise ValueError('VOUCHER_USED')
        if subtotal <= v.min_subtotal: raise ValueError('MIN_SUBTOTAL_NOT_MET')
        return min(money(subtotal*v.percent/D(100)),v.cap)


class Shipping:
    def calculate(self,subtotal,area):
        if area not in ('inner','outer'): raise ValueError('INVALID_AREA')
        return D(0) if area=='inner' and subtotal>=D(500000) else (D(30000) if area=='inner' else D(50000))

class Payments:
    def __init__(self): self.charges=[]
    def collect(self,amount,method,approved,key):
        if method=='COD': return 'PENDING_COD'
        if method!='ONLINE': raise ValueError('INVALID_PAYMENT_METHOD')
        if not approved: raise ValueError('PAYMENT_DECLINED')
        self.charges.append({'key':key,'amount':amount})
        return 'PAID'

class Checkout:
    def __init__(self, catalog=None):
        self.catalog = catalog or Catalog()
        self.inventory = Inventory(self.catalog)
        self.promotions = Promotions()
        self.shipping = Shipping()
        self.payments = Payments()
        self.orders = {}
        self.by_key = {}
        self.counter = 0

    def calculate_total(self, subtotal, discount, shipping):
        return max(D(0), subtotal - discount + shipping)

    def place(self, customer, cart, area='inner',
              voucher=None, method='COD',
              approved=True, key='REQ-001'):

        if not customer.authenticated:
            raise ValueError('LOGIN_REQUIRED')

        if not cart.lines:
            raise ValueError('EMPTY_CART')

        # 1. Check idempotency .
        if key in self.by_key:
            return self.orders[self.by_key[key]]

        # 2. Calculate order amount.
        subtotal = cart.subtotal()
        discount = self.promotions.discount(
            voucher, subtotal, customer
        )
        shipping = self.shipping.calculate(subtotal, area)
        total = max(D(0), subtotal - discount + shipping)

        # 3. Reserve inventory.
        self.inventory.reserve(cart.lines)

        # 4. Process payment.
        try:
            payment_state = self.payments.collect(
                total, method, approved, key
            )
        except Exception:
            # Release stock if payment processing fails.
            self.inventory.release(cart.lines)
            raise

        # 5. Create order.
        self.counter += 1
        order = Order(
            f'ORD-{self.counter:04d}',
            customer.customer_id,
            total,
            payment_state,
            [(x.sku, x.quantity) for x in cart.lines],
            discount,
            shipping,
            method
        )

        # 6. Save order and idempotency key.
        self.orders[order.order_id] = order
        self.by_key[key] = order.order_id

        if voucher:
            customer.used_codes.add(voucher)

        return order

def fixture(fixture_name):
    service = Checkout()
    customer = Customer("U_TEST", authenticated=True)
    cart = Cart(service.catalog)

    if fixture_name == 'STANDARD':
        cart.add('A', 2)

    elif fixture_name == 'MULTI_ITEM':
        cart.add('A', 1)
        cart.add('B', 1)

    elif fixture_name == 'ANONYMOUS_USER':
        customer.authenticated = False
        cart.add('A', 1)

    elif fixture_name == 'EMPTY_CART':
        pass

    elif fixture_name == '600k':
        cart.add('A', 3)

    elif fixture_name == '300k':
        cart.add('B', 1)

    else:
        raise ValueError(
            f'UNKNOWN_FIXTURE: {fixture_name!r}'
        )

    return service, customer, cart


Overwriting ecommerce_app.py


In [ ]:
import ecommerce_app
import importlib
importlib.reload(ecommerce_app)

<module 'ecommerce_app' from '/content/ecommerce_app.py'>

In [ ]:
%%writefile TestCases.csv
case_id,requirement_id,condition_id,scenario_id,fixture,operation,input_data,expected_type,expected_value,expected_error,priority,precondition,steps
TC-CRT-001,REQ-02,COND-CRT-01,TS-CRT-01,STANDARD,cart_add,"{""sku"":""A"",""qty"":2}",VALUE,4,,P1,Cart initialized with 2 SKU A,Add 2 SKU A to cart -> Verify total quantity is 4
TC-CRT-002,REQ-02,COND-CRT-02,TS-CRT-02,STANDARD,cart_add,"{""sku"":""A"",""qty"":0}",EXCEPTION,ERROR:INVALID_QUANTITY,INVALID_QUANTITY,P0,Cart initialized,Add 0 quantity -> Expect INVALID_QUANTITY
TC-CRT-003,REQ-02,COND-CRT-02,TS-CRT-02,STANDARD,cart_add,"{""sku"":""A"",""qty"":-3}",EXCEPTION,ERROR:INVALID_QUANTITY,INVALID_QUANTITY,P0,Cart initialized,Add -3 quantity -> Expect INVALID_QUANTITY
TC-CRT-004,REQ-02,COND-CRT-01,TS-CRT-01,STANDARD,cart_add_accumulate,"{""sku"":""A"",""qty"":3}",VALUE,5,,P1,Cart has SKU A with qty 2,Add 3 more SKU A -> Verify accumulated qty is 5
TC-PRC-001,REQ-03,COND-PRC-01,TS-PRC-01,MULTI_ITEM,cart_subtotal,"{}",VALUE,1200000,,P1,Cart has SKU A (200k) and SKU B (300k),Calculate cart subtotal -> Verify subtotal is 1200000
TC-PRC-002,REQ-03,COND-PRC-02,TS-PRC-02,STANDARD,cart_subtotal_type,"{}",TYPE,Decimal,,P1,Cart initialized with lines,Check return type of cart.subtotal() -> Verify it is Decimal
TC-ATH-001,REQ-04,COND-ATH-01,TS-ATH-01,STANDARD,checkout_place,"{""method"":""COD""}",VALUE,PENDING_COD,,P1,Authenticated user with lines in cart,Place order -> Verify status is PENDING_COD
TC-ATH-002,REQ-04,COND-ATH-02,TS-ATH-02,ANONYMOUS_USER,checkout_place,"{""method"":""COD""}",EXCEPTION,ERROR:LOGIN_REQUIRED,LOGIN_REQUIRED,P0,Anonymous user with lines in cart,Place order -> Expect LOGIN_REQUIRED exception
TC-ATH-003,REQ-04,COND-ATH-02,TS-ATH-02,ANONYMOUS_USER,checkout_place_expired,"{""method"":""COD""}",EXCEPTION,ERROR:LOGIN_REQUIRED,LOGIN_REQUIRED,P0,User session expired,Place order -> Expect LOGIN_REQUIRED exception
TC-ORD-001,REQ-12,COND-ORD-01,TS-ORD-01,MULTI_ITEM,checkout_idempotency,"{""idempotency_key"":""KEY_ABC_123"",""method"":""ONLINE""}",VALUE,IDEMPOTENT_OK,,P0,Cart has valid lines and user is authenticated,Send duplicate checkout requests with same idempotency_key -> Verify system returns the exact same order_id and prevents duplicate creation
TC-ORD-002,REQ-12,COND-ORD-02,TS-ORD-02,MULTI_ITEM,calculate_total,"{""area"":""inner"",""voucher"":""SAVE10""}",VALUE,450000,,P1,Cart subtotal 500k; inner ship (0k); SAVE10 (50k),Place order -> Verify total = 500k - 50k + 0k = 450000
TC-ORD-003,REQ-12,COND-ORD-03,TS-ORD-02,STANDARD,calculate_total_boundary,"{""subtotal"":100000,""discount"":200000}",VALUE,0,,P0,Discount exceeds subtotal,Calculate total -> Verify total floors at 0
TC-CRI-001,REQ-14,COND-CRI-01,TS-CRI-02,STANDARD,checkout_place,"{""method"":""COD""}",VALUE,PENDING_COD,,P1,Cart has valid lines (>=1),Place order -> Succeeds normally
TC-CRI-002,REQ-14,COND-CRI-02,TS-CRI-01,EMPTY_CART,checkout_place,"{""method"":""COD""}",EXCEPTION,ERROR:EMPTY_CART,EMPTY_CART,P0,Cart is empty (0 lines),Place order -> Expect EMPTY_CART exception
TC-CRI-003,REQ-14,COND-CRI-02,TS-CRI-01,EMPTY_CART,checkout_place_online,"{""method"":""ONLINE"",""approved"":true}",EXCEPTION,ERROR:EMPTY_CART,EMPTY_CART,P0,Cart is empty (0 lines),Place online order -> Expect EMPTY_CART exception

Writing TestCases.csv


In [ ]:
%%writefile test_runner.py
"""CSV-driven execution, requirement traceability and Excel/HTML reporting."""
from pathlib import Path
import csv, datetime, html, json
from collections import Counter
from decimal import Decimal
import traceback

import pandas as pd
from ecommerce_app import fixture

ROOT=Path(__file__).resolve().parent

def parse_input_data(input_str):
    if not input_str or str(input_str).strip() in ("", "{}"):
        return {}

    try:
        data = json.loads(str(input_str))
    except json.JSONDecodeError as exc:
        raise ValueError(f"INVALID_INPUT_JSON: {exc}")

    if not isinstance(data, dict):
        raise ValueError("INPUT_DATA_MUST_BE_OBJECT")

    return data

def execute_case(row):
    app,user,cart=fixture(row['fixture'])
    op=row['operation']; actual_error=''; actual=''; order=None
    before={k:p.stock for k,p in app.catalog.products.items()}

    input_data = parse_input_data(row.get('input_data', '{}'))
    try:
        if op=='voucher':
            order=app.place(user,cart,voucher=row.get('voucher') or None)
            actual=f'{order.discount:.0f}'
        elif op=='payment_failure':
            app.place(user,cart,method='ONLINE',approved=False)
            actual='NO_ERROR'
        elif op=='payment_success':
            order=app.place(user,cart,method='ONLINE',approved=True)
            actual=order.status
        elif op=='stock_failure':
            for p in app.catalog.products.values(): p.stock=0
            app.place(user,cart); actual='NO_ERROR'
        elif op == 'retry':
            idempotency_key = input_data.get(
                'idempotency_key', row['case_id']
            )

            order1 = app.place(
                user, cart, key=idempotency_key
            )

            order2 = app.place(
                user, cart, key=idempotency_key
            )

            actual = str(len(app.orders))
        elif op == 'cart_add' or op == 'cart_add_accumulate':
            sku = input_data.get('sku')
            qty = int(input_data.get('qty', 0))
            cart.add(sku, qty)
            actual = str(cart.total_quantity())
        elif op == 'cart_subtotal':
            lines = input_data.get('lines', [])
            if lines:
                subtotal = sum(Decimal(str(item['price'])) * int(item['qty']) for item in lines)
                actual = str(int(subtotal))
            else:
                actual = str(int(cart.subtotal()))

        elif op == 'cart_subtotal_type':
            subtotal = cart.subtotal()
            actual = type(subtotal).__name__

        elif op == 'checkout_place':
            method = input_data.get('method', 'COD')
            order = app.place(user, cart, method=method)
            actual = order.status

        elif op == 'checkout_place_expired':
            user.authenticated = False
            method = input_data.get('method', 'COD')
            order = app.place(user, cart, method=method)
            actual = order.status

        elif op == 'checkout_idempotency':
            idempotency_key = input_data.get('idempotency_key')

            if not idempotency_key:
                raise ValueError('MISSING_IDEMPOTENCY_KEY')

            method = input_data.get('method', 'ONLINE')

            order1 = app.place(
                user, cart,
                method=method,
                key=idempotency_key
            )

            order2 = app.place(
                user, cart,
                method=method,
                key=idempotency_key
            )

            actual = (
                "IDEMPOTENT_OK"
                if order1.order_id == order2.order_id
                else "IDEMPOTENT_FAIL"
            )

        elif op == 'calculate_total':
            area = input_data.get('area', 'inner')
            voucher = input_data.get('voucher')
            order = app.place(user, cart, method='COD', area=area, voucher=voucher)
            actual = str(int(order.total))

        elif op == 'calculate_total_boundary':
            subtotal = Decimal(str(input_data.get('subtotal', 0)))
            discount = Decimal(str(input_data.get('discount', 0)))
            shipping = Decimal(str(input_data.get('shipping', 0)))

            total = app.calculate_total(subtotal, discount, shipping)
            actual = str(int(total))

        elif op == 'checkout_place_online':
            approved = input_data.get('approved', True)
            method = input_data.get('method', 'ONLINE')
            order = app.place(user, cart, method=method, approved=approved, key=row['case_id'])
            actual = order.status

        else:
            raise ValueError('UNKNOWN_TEST_OPERATION')
    except ValueError as exc:

        actual_error = str(exc)
        actual = f'ERROR:{actual_error}'

    except Exception as exc:


        actual_error = f'UNEXPECTED_{type(exc).__name__}: {exc}'
        actual = f'ERROR:{actual_error}'
    after={k:p.stock for k,p in app.catalog.products.items()}
    expectation = str(row.get('expected_value', ''))
    actual_str = str(actual)

    assertion_type = row.get('assertion', 'equals')
    expected_error = str(row.get('expected_error', '')).strip()

    if not assertion_type or assertion_type == 'equals':
      if actual_error:
          passed = actual_error == expected_error
      else:
          passed = actual_str == expectation

    elif assertion_type == 'stock_unchanged':
        actual_str = 'UNCHANGED' if before == after else 'CHANGED'
        passed = (actual_str == expectation and actual_error == row.get('expected_error', ''))
    elif assertion_type == 'retry_stock_once':
        expected_stock = dict(before)

        for line in cart.lines:
            expected_stock[line.sku] -= line.quantity

        stock_ok = (after == expected_stock)
        order_ok = (len(app.orders) == 1)

        actual_str = (
            'ONCE'
            if stock_ok and order_ok
            else 'MULTIPLE'
        )

        passed = (
            actual_str == expectation
            and order_ok
            and len(app.payments.charges) <= 1
        )
    else:
        passed = (actual_str == expectation)
    return {**row,'actual_value':actual,'actual_error':actual_error,'status':'PASS' if passed else 'FAIL',
            'observed_before':json.dumps(before),'observed_after':json.dumps(after),
            'executed_at':datetime.datetime.now(datetime.timezone.utc).isoformat()}

def run_suite(csv_path=None,out_dir=None):
    csv_path=Path(csv_path or ROOT/'TestCases.csv')
    output=Path(out_dir or ROOT/'reports'); output.mkdir(exist_ok=True,parents=True)
    with csv_path.open(newline='',encoding='utf-8-sig') as f: testcases=list(csv.DictReader(f))
    ids=[x['case_id'] for x in testcases]
    if len(ids)!=len(set(ids)): raise ValueError('DUPLICATE_TEST_IDS')
    results=[execute_case(x) for x in testcases]
    df=pd.DataFrame(results); summary=df.groupby(['requirement_id','status']).size().unstack(fill_value=0).reset_index()
    counts=Counter(df.status)
    totals=pd.DataFrame([{'executed':len(df),'passed':counts['PASS'],'failed':counts['FAIL'],'pass_rate':round(100*counts['PASS']/len(df),1)}])
    df.to_csv(output/'test_execution.csv',index=False,encoding='utf-8-sig')
    with pd.ExcelWriter(output/'test_report.xlsx',engine='openpyxl') as xls:
        df.to_excel(xls,index=False,sheet_name='Execution')
        summary.to_excel(xls,index=False,sheet_name='Traceability')
        totals.to_excel(xls,index=False,sheet_name='Summary')
    rows=''.join('<tr>'+''.join(f'<td>{html.escape(str(row.get(key,"")))}</td>' for key in ['case_id','requirement_id','scenario_id','expected_value','actual_value','status'])+'</tr>' for row in results)
    page='''<!DOCTYPE html><html><head><meta charset="utf-8"><title>EduCommerce Test Report</title><style>body{font:15px Arial;margin:35px;color:#14233c}h1{color:#17385f}table{border-collapse:collapse;width:100%}td,th{border:1px solid #cbd5e1;padding:9px;text-align:left}th{background:#16385e;color:white}tr:nth-child(even){background:#f1f5f9}.metric{display:inline-block;margin:12px;padding:16px;border-radius:12px;background:#edf3f9;font-size:22px}</style></head><body><h1>EduCommerce – Automated Test Execution</h1>'''+f'<div class="metric">Total: {len(df)}</div><div class="metric">PASS: {counts["PASS"]}</div><div class="metric">FAIL: {counts["FAIL"]}</div><h2>Traceability & Execution</h2><table><tr><th>Case</th><th>Requirement</th><th>Scenario</th><th>Expected</th><th>Actual</th><th>Status</th></tr>{rows}</table></body></html>'
    (output/'test_dashboard.html').write_text(page,encoding='utf-8')
    print(f'Executed {len(df)} | PASS {counts["PASS"]} | FAIL {counts["FAIL"]}')
    print(df.loc[df.status=='FAIL',['case_id','expected_value','actual_value']].to_string(index=False))
    return df,output

if __name__=='__main__': run_suite()


Overwriting test_runner.py


In [ ]:
import importlib
import test_runner

importlib.reload(test_runner)

print("Loaded file:", test_runner.__file__)

Loaded file: /content/test_runner.py


In [ ]:
!pip -q install pandas openpyxl matplotlib gradio
from test_runner import run_suite
results, report_dir = run_suite("TestCases.csv", "reports")
print(results[["case_id","requirement_id","expected_value","actual_value","status"]].to_string(index=False))


Executed 15 | PASS 13 | FAIL 2
   case_id expected_value               actual_value
TC-PRC-001        1200000                     500000
TC-ORD-002         450000 ERROR:MIN_SUBTOTAL_NOT_MET
   case_id requirement_id         expected_value               actual_value status
TC-CRT-001         REQ-02                      4                          4   PASS
TC-CRT-002         REQ-02 ERROR:INVALID_QUANTITY     ERROR:INVALID_QUANTITY   PASS
TC-CRT-003         REQ-02 ERROR:INVALID_QUANTITY     ERROR:INVALID_QUANTITY   PASS
TC-CRT-004         REQ-02                      5                          5   PASS
TC-PRC-001         REQ-03                1200000                     500000   FAIL
TC-PRC-002         REQ-03                Decimal                    Decimal   PASS
TC-ATH-001         REQ-04            PENDING_COD                PENDING_COD   PASS
TC-ATH-002         REQ-04   ERROR:LOGIN_REQUIRED       ERROR:LOGIN_REQUIRED   PASS
TC-ATH-003         REQ-04   ERROR:LOGIN_REQUIRED       ERROR:LO

In [ ]:
import pandas as pd
summary = results.groupby(["requirement_id","status"]).size().unstack(fill_value=0)
print(summary)
print("Report folder:", report_dir)


status          FAIL  PASS
requirement_id            
REQ-02             2     2
REQ-03             2     0
REQ-04             1     2
REQ-12             3     0
REQ-14             1     2
Report folder: reports


In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

report_df = pd.read_csv("reports/test_execution.csv").fillna("")

def dashboard_view(req="ALL", status="ALL"):
    df = report_df.copy()
    if req != "ALL": df = df[df["requirement_id"] == req]
    if status != "ALL": df = df[df["status"] == status]
    summary = df["status"].value_counts().reindex(["PASS", "FAIL"], fill_value=0)
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(summary.index, summary.values, color=["#16a34a", "#dc2626"])
    ax.bar_label(bars, padding=3)
    ax.set_title("Test execution results")
    ax.set_ylabel("Test cases")
    ax.set_ylim(0, max(2, int(summary.max()) + 2))
    fig.tight_layout()
    cols = ["case_id", "requirement_id", "scenario_id", "expected_value", "actual_value", "status"]
    return fig, df[cols], f"Executed: {len(df)} | PASS: {summary['PASS']} | FAIL: {summary['FAIL']}"

with gr.Blocks(title="EduCommerce - Test Dashboard") as demo:
    gr.Markdown("# EduCommerce — Requirement Traceability & Test Results")
    with gr.Row():
        req_filter = gr.Dropdown(["ALL"] + sorted(report_df.requirement_id.unique().tolist()), value="ALL", label="Requirement")
        status_filter = gr.Dropdown(["ALL", "PASS", "FAIL"], value="ALL", label="Test status")
    status_line = gr.Markdown()
    with gr.Row(equal_height=True):
        with gr.Column(scale=1, min_width=300):
            chart = gr.Plot(label="Result distribution")
        with gr.Column(scale=2, min_width=420):
            result_table = gr.Dataframe(label="Test execution details", interactive=False, wrap=True)
    gr.File(value="reports/test_report.xlsx", label="Download Excel report")
    req_filter.change(dashboard_view, [req_filter, status_filter], [chart, result_table, status_line])
    status_filter.change(dashboard_view, [req_filter, status_filter], [chart, result_table, status_line])
    demo.load(dashboard_view, [req_filter, status_filter], [chart, result_table, status_line])

# share=True yêu cầu link tạm (*.gradio.live), không phải URL cố định.
# Không dùng share=True với dữ liệu nhạy cảm.
demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a2bf07832345ef8ef5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import files
files.download("reports/test_report.xlsx")
# Tùy chọn: files.download("reports/test_execution.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>